[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enkaoua/coding-with-ai-tutorial/blob/main/using_AI_tutorial_STUDENT.ipynb)

# How to use AI properly 

Imagine asking an AI for a bread recipe and getting one with broccoli in it, you’d spot that something is off straight away (hopefully....?). 
Now imagine it quietly gets the ratio of flour to water wrong. The recipe still looks normal, and you only find out something was wrong when the bread comes out like a brick. That is... unless you know enough about bread to tell when you mixed the dough if it was too dry or too wet.

Code can fail in the same way, where it can either give an error (like broccoli in bread) OR.. more subtly, it can run, the plot can look believable, but the numbers are quietly wrong. AI can be a great helper, but only if you know enough of the basics to tell when something is off.

In [ ]:
#@title Setup: run this cell first (it downloads the code that checks your answers)
import os, sys

# Colab only: download this tutorial's helper code (tutorial_helpers.py: the answer checkers and the simulated board) from GitHub.
# On your own computer tutorial_helpers.py is already next to this notebook, so nothing is downloaded.
REPO_URL = "https://github.com/enkaoua/coding-with-ai-tutorial.git"

if "google.colab" in sys.modules:
    REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

import numpy as np
import matplotlib.pyplot as plt

try:
    from tutorial_helpers import *
except ModuleNotFoundError:
    raise ModuleNotFoundError("tutorial_helpers.py was not found next to this notebook. "
                              "On your own computer, open the whole tutorial folder (in VS Code: File > Open Folder), "
                              "not just the notebook file.") from None

print("✅ Setup done. Carry on with the notebook.")

## Understand what you are doing

### understand the basics
When you mix the dough for bread, you should know what sort of dough consistency you’re looking for. If it doesn’t combine, it probably needs more liquid. If it’s too wet, it needs more flour. You can fix it because you know what you’re testing for. It’s the same with code. To test it, and to fix it when it fails, you need to understand what it should be doing. So before we write anything, let’s make sure we understand the basics of the problem we want to solve.

### today's task
You’ve already built the ECG circuit, which picks up the heart’s small electrical signals and amplifies them into a voltage that changes over time. Today we’re focusing on the next step: data acquisition and display. A computer can’t read a voltage directly, it only understands numbers. So between the circuit and the computer sits an ADC (analogue-to-digital converter). It measures the voltage many times a second and turns each measurement into a number. Our job today is to read those numbers, and turn them back into a correct picture of the heart’s signal.

### ADC basics
An ADC can’t store an exact voltage like “2.5 V”. Instead, it splits its input range into a fixed number of steps, and tells you which step the voltage falls into. That step number is called a count.

The board we'll use has:
- a **10-bit** ADC, which means 2¹⁰ = 1024 steps (counts 0 to 1023)
- an input range of **0 to 5 V**

**Try it before reading on:**
1. Roughly how big is one step, in volts?
2. If the ADC returns 512, about how many volts is that?

In [ ]:
# 1. Roughly how big is one step, in volts?
step_size = None       # your answer, in volts
# 2. If the ADC returns 512, about how many volts is that?
volts_for_512 = None    # your answer, in volts

check_adc_basics(step_size, volts_for_512)

### sampling: how often do we measure?
The ADC doesn't measure continuously. It takes a snapshot of the voltage, waits a fixed time, and takes another snapshot. The number of snapshots per second is the **sampling rate**, called `fs`, measured in samples per second (Hz).

For example, `fs = 250` means 250 snapshots every second.

You can think of it like a camera taking a video. If it takes 30 pictures per second, you see smooth motion. If it takes only 1 picture per second, you see a series of stills.

**Try it before reading on:**
If our sampling rate is 250 Hz:
1. How many seconds pass between one sample and the next?
2. If we take 500 samples at this rate, how long is the recording?

In [ ]:
# 1. How many seconds pass between one sample and the next?
spacing = None       # your answer, in seconds
# 2. If we take 500 samples at 250 samples per second, how long is the recording?
duration = None      # your answer, in seconds

check_sampling_basics(spacing, duration)

## Break things down and test them

Now we have the basics, let’s use them. Imagine asking an AI: “write me a program that reads my ECG and plots it.” It gives you a long program in one go, and it might even run. But now you have to read the whole thing to check it, and you don’t know where anything is: which lines read the data, which convert it to volts, which draw the plot. If the plot looks wrong, you don’t know where to start looking. If it looks right but a number is quietly wrong, you may never notice. You can ask the AI to fix it, but you can only ask for a fix if you can describe the problem, and you can’t describe a problem if you don’t know where it is or what the right answer should look like. And if the fix breaks something else, you’re back to reading the whole program again.

Instead, we'll break the project into small parts, and for each part, do the same four things:

1. **Decide what "right" looks like.** Before writing any code, work out how you'll know it's correct.
2. **Learn.** Ask the AI to explain the idea or quiz you, not to write the answer.
3. **Plan.** Write your steps down in a way that works for you, with a check for each one.
4. **Build, then check.** Write the code yourself, or have the AI turn your plan into code. Then run your tests, and make sure you can explain every line.

### 1. A worked example: decide what "right" looks like



Let's practise the method on a small task. 

**The task:** write a function `calculate(a, b, operation)` that takes two numbers and an operation (add, subtract, multiply or divide), and returns the result.

Before any code exists, decide how you'll test it. Good tests include **normal cases** (does it work?) and **edge cases** (does it break?). Edge cases are the awkward ones: zero, negative numbers, and inputs that don't make sense.

**Try it before reading on:** fill in what you expect each call to give.

| Call | What I expect |
|---|---|
| `calculate(6, 3, "add")` | |
| `calculate(6, 3, "subtract")` | |
| `calculate(6, 3, "multiply")` | |
| `calculate(6, 3, "divide")` | |
| `calculate(5, 0, "add")` | |
| `calculate(5, 0, "multiply")` | |
| `calculate(0, 5, "divide")` | |
| `calculate(-2, 3, "multiply")` | |
| `calculate(5, 0, "divide")` | |
| `calculate(5, 3, "power")` | |

The last two are different from the rest. What *should* the function do? You decide.

The first eight are 9, 3, 18, 2, 5, 0, 0 and −6. Zero is only a problem when you divide by it. Adding, multiplying and dividing zero by something are all fine.

The last two have no single right answer: you decide. A function could stop with an error, print a message, or return a message. What matters is that the behaviour is a clear choice you made, not Python's own crash.

### 2. learn

In this example, the problem is so simple that there isn't much to learn first. But in most real problems, this is the step where you learn the background: what the idea is, what the terms mean, and what could go wrong. (For the ECG, that was the ADC and sampling basics at the start.)

Even here, the AI can help you think. Try this prompt:

> *"I want to write a Python function that adds, subtracts, multiplies or divides two numbers. Don't write any code. What could go wrong, and what edge cases should I test?"*

Did it mention anything you hadn't thought of? Would you add it to your tests?

### 3. write your plan in a way that works for you


Now we plan it. There's no single right way... it depends on your preference! Some people like a table, some like a flowchart, and some like to write the steps out like code (called **pseudocode**) first. Here's the same `calculate` task in three styles.

**1. Table**

| Step | How I'll know it's right |
|---|---|
| Check the operation is one we know | An unknown operation like "power" gives a clear error message |
| Do the operation | 6 and 3 gives 9, 3, 18 and 2 for add, subtract, multiply, divide |
| Handle dividing by zero | 5 ÷ 0 gives a clear message, not a crash |
| Return the result | The function gives back a number |

**2. Flowchart**

    start
      ↓
    is the operation one we know? ── no ──► error message
      ↓ yes
    which operation?
      ├─ add ──────► a + b ───────────────┐
      ├─ subtract ─► a − b ───────────────┤
      ├─ multiply ─► a × b ───────────────┤
      └─ divide ──► is b zero? ─ yes ─► error message
                       ↓ no
                     a ÷ b ───────────────┤
                                          ↓
                                 return the result

**3. Pseudocode** (plain-English "code")

    calculate(a, b, operation):
        if operation is "add":        result = a + b
        else if operation is "subtract": result = a − b
        else if operation is "multiply": result = a × b
        else if operation is "divide":
            if b is zero: report an error     # CHECK: does not crash
            else: result = a ÷ b
        else: report an error                 # CHECK: unknown operation
        return result

Pick whichever style suits you. **Whichever you choose, every step needs a way to check it.** A plan without checks is just a to-do list.

### 4. let AI turn your plan into code

Give your AI the pseudocode and ask it to follow it exactly:

> *"I am writing a function that takes two numbers and an operation, and performs the operation on those two numbers. Could you write me a Python function that follows this pseudocode: [.... paste your pseudocode/plan here] Don't add anything that isn't in the plan. Then explain each part in plain English."*

Paste what it gives you below, and run the tests.

In [ ]:
# Paste the AI's function here, replacing the def below. Keep the last line: it runs the tests.
def calculate(a, b, operation):
    return None

test_calculate(calculate)

### 5. read it against your plan

Passing the tests isn't enough. Read the code next to your pseudocode:

1. Does every line of code match a step in your plan?
2. Is there anything in the code that **isn't** in your plan?
3. How does it report errors? Is that what you wanted?
4. Explain each part back in one sentence.

## Now you try it: prime numbers

Same method, simpler problem. Before the ECG, let's practice on something where you already know what "right" looks like.

**The task:** write a function `is_prime(n)` that returns `True` if `n` is a prime number, and `False` if it isn't.

### 1. decide what "right" looks like
Before any code, write down what you expect. **Try it before reading on:**
1. Which numbers below 30 are prime?
2. How many primes are there below 100?
3. Is 0 prime? Is 1? Is a negative number?

### 2. learn: ask your AI to explain, not to write

Try this prompt:

> *"I want to write a Python function that checks whether a number is prime. Don't write any code. Explain the idea in plain English, then ask me one question to check I've understood."*

Answer its question. Did it correct you?

### 3. plan it

Write your plan in whichever style you like (table, flowchart or pseudocode), with a way to check every step.

If it feels too big, **make it smaller**: how would you check whether just one number, say 7 or 9, is prime? Get that working, then wrap it in a function.

A useful tool: in Python, `a % b` gives the **remainder** when you divide a by b. For example, `9 % 3` is `0`, and `9 % 2` is `1`.

*Type out your plan here:*

...




### 4. build it, then check it

Write your function below. Start small, get it working for the numbers below 30, then run the test.

In [ ]:
def is_prime(n):
    # TODO: write your function here
    return None

test_is_prime(is_prime)

### 5. now compare with AI's version

Here's the kind of code an AI might give you. It's a **faster** version: instead of checking every number up to `n`, it only checks up to the square root of `n`. Run the same test on it.

In [ ]:
def is_prime_ai(n):
    if n < 2:
        return False
    for divisor in range(2, int(n ** 0.5)):
        if n % divisor == 0:
            return False
    return True

test_is_prime(is_prime_ai)

Look at the results.

1. Which numbers does it get wrong?
2. Would you have spotted the problem just by reading the code?
3. Can you find the line responsible? (Hint: look at the `range`.)
4. What would you ask the AI to fix it, and how would you check the fix?

## Plan the ECG project

Our project: *read an ECG from the board and display it as a correct waveform, with time in seconds and voltage in volts, then clean it and measure the heart rate.*

Write down the small steps you'd split it into (aim for 4 to 6). For each one, write how you'd know it's correct.
Use any style from before: a table (like below), a flowchart, or pseudocode.

| Step | How I'll know it's right |
|---|---|
| 1. .... | ... |
| 2. | |
| 3. | |
| 4. | |
| 5. | |
| 6. | |

### one way to break it down

| Step | How I'll know it's right |
|---|---|
| Read the raw counts | They're whole numbers between 0 and 1023 |
| Build the time axis | The first sample is at 0 s and the spacing is 1/fs |
| Convert counts to volts | A known 2.000 V reference reads 2.000 V |
| Plot it | The baseline sits near mid-range and a beat appears every 0.6 to 1 s (a resting heart is 60 to 100 bpm) |
| Remove the 50 Hz hum | The 50 Hz ripple is gone and the R wave is still as tall as before |
| Find the heart rate | It's plausible for a resting heart (60 to 100 bpm) and matches counting the beats by hand |

Your list may look different, and that's fine. What matters is that **every step has a way to check it**.

### what if you can't break it down any further?

Sometimes a step is still too big to split. In that case, **make the project smaller** instead:

- one channel instead of three (eg if you're dealing with RGB images)
- 2 seconds of data instead of a minute
- a simple signal where you already know the right answer, before the real ECG

Get the small version working and checked first, then grow it. If something breaks, you only changed one thing since the last time it worked.

## The board and the setup

We don’t have hardware in the room today, so the board is simulated in Python. For your code it behaves like the real thing, you ask the function for readings, and it gives you raw counts back.

It's modelled on the ADC in the Arduino Uno's chip, the **ATmega328P**, so the ADC details match what you'll meet in the lab. (The three channels below only exist in our simulation.)

You can look at the real datasheet for the real values, but here are the most important ones:


| | |
|---|---|
| ADC | 10-bit |
| Input range | 0 to 5 V |
| Channel 0 | the ECG, after the amplifier |
| Channel 1 | a test tone from a function generator |
| Channel 2 | a reference wired to exactly 2.000 V |

Channel 2 is there for testing. We know exactly what voltage it carries, so if our code says anything different, we know the code is wrong, not the board.

To ask the board for readings, we'll use `read_adc(channel, n_samples, fs)`: which channel, how many samples, and how fast. Run the cell below to turn the board on.

In [ ]:
daq = SimulatedDAQ(seed=42)
print("Board is on.")

## Part 1: Read the raw counts

**The goal:** get readings from the board into Python.

Same method as before. We've already done the *learn* step for this part (the ADC basics at the start), so let's start by deciding what "right" looks like.

As you learnt in the last lab, the ECG circuit lifts the signal up to the middle of the supply, 2.5 V, so the heartbeat can swing both above and below it.

**Try it before reading on:**
1. When the heart is resting between beats, roughly what count do you expect the board to give? (You already worked out the count for 2.5 V.)
2. A resting heart beats 60 to 100 times a minute. If we record 500 samples at 250 Hz (2 seconds), roughly how many beats should we see?

In [ ]:
# 1. When the heart is resting, roughly what count do you expect?
baseline_guess = None     # your answer, a count (0 to 1023)
# 2. Roughly how many beats should we see in 2 seconds?
beats_guess = None       # your answer

check_part1_predictions(baseline_guess, beats_guess)

Let's start small (remember: make it smaller!). Ask the board for just 5 readings from channel 0, at 250 samples per second:

In [ ]:
daq.read_adc(channel=0, n_samples=5, fs=250)

Look at the numbers. They're whole numbers (counts), and they're close to the value you predicted.

Run the cell above again. Do you get the same five numbers? Why not?

Now let's read the full recording, 500 samples, and plot it.

In [ ]:
fs = 250            # samples per second
n_samples = 500     # how many samples to read

counts = daq.read_adc(channel=0, n_samples=n_samples, fs=fs)

plt.figure(figsize=(9, 3))
plt.plot(counts)
plt.xlabel("sample number")
plt.ylabel("ADC counts")
plt.show()

Compare the plot with your predictions:

1. Where do the flat parts of the line sit? Is that where you expected?
2. How many beats can you count? Is that what you expected?
3. Look at the x-axis: it says "sample number". Could you tell from this plot how long one heartbeat takes, in seconds?

## Part 2: The time axis

**The goal:** make an array `t` that holds the time, in seconds, of every sample. The board gives us numbers, but not *when* each one happened. That comes from `fs`.

**1. Decide what "right" looks like.** Work these out **by hand**, for 500 samples at 250 samples per second. Check your units, and remember that the very first sample is at t = 0.

**Try it before reading on:**
1. How many seconds are there between one sample and the next?
2. At what time does the **last** sample (the 500th) happen?

In [ ]:
# 1. How many seconds between one sample and the next?
expected_spacing = None             # your answer, in seconds
# 2. At what time does the last sample happen?
expected_last_time = None   # your answer, in seconds

check_time_predictions(expected_spacing, expected_last_time)

**2. Learn.** Ask your AI to explain, not to write code:

> *"I read 500 samples at 250 Hz. Without writing any code, explain how to work out the time of each sample, then ask me what time the last sample is."*

Answer its question. Does its answer match yours? If not, who is right, and how would you decide?

**3. Plan.** This one is small enough for a one-line plan. Write it in whichever style you like. Your check is already written: the two numbers you worked out above.

*Write your plan below:*

**4. Build, then check.** Make `t` with `numpy`. Two useful tools: `np.arange(n)` gives `0, 1, 2, ... n-1`, and you can divide a whole array by a number. (`fs` and `n_samples` are still set from Part 1.)

In [ ]:
# Build the time axis: one time (in seconds) for each sample
t = None     # replace None with your code (use n_samples and fs)

check_time_axis(t, expected_spacing, expected_last_time)

**5. Now compare with AI's version.** An AI might well give you this to build the time axis, because "500 samples over 2 seconds" sounds right:

    t_ai = np.linspace(0, 2, 500)

Don't judge it by eye. Run the same check on it.

In [ ]:
t_ai = np.linspace(0, 2, 500)

check_time_axis(t_ai, expected_spacing, expected_last_time)

Look at the results.

1. What's wrong with it?
2. Would you have spotted it on a plot?
3. How would you fix it, and how would you check the fix?
4. Explain it back in one sentence: why is the last sample at 1.996 s and not 2.000 s?

*Write your answers here:*



In [ ]:
#@title Stuck? Run this cell to carry on with a working time axis (try it yourself first!)
t = np.arange(500) / 250

Now we can plot the ECG against real time:

In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(t, counts)
plt.xlabel("time (s)")
plt.ylabel("ADC counts")
plt.show()

## Part 3: Counts → volts

**The goal:** turn the raw counts into volts. Right now our plot shows counts, not a voltage.

**1. Decide what "right" looks like.** Remember channel 2, the reference wired to exactly 2.000 V? We know what it should read, so it's our test: whatever code we write must turn the channel 2 counts into 2.000 V. (We allow a tiny bit of wiggle room for noise and for the ADC's step of about 5 mV.)



**2. Learn.** Ask your AI to explain, not to write code. This time, tell it about our board:

> *"I have a 10-bit ADC with a 0 to 5 V input range. Don't write any code. Explain how to convert a count to a voltage, then ask me what voltage a count you give me is."*



**3. Plan.** Write your plan in whichever style you like. What goes in, what comes out, and what is the one calculation in between?

*Write your plan below:*


**4. Build, then check.** Write a function `counts_to_volts(counts)` that takes the raw counts and returns the voltage for each one. `counts` is a whole array, so you can calculate on all of it at once. The board's range and number of steps are in the data sheet table in the "board and setup" section.

In [ ]:
def counts_to_volts(counts):
    # TODO: turn the raw counts into volts
    return None

check_counts_to_volts(counts_to_volts)

**5. Now compare with AI's version.** Here's the kind of code an AI might give you if you ask it "write a function that converts ADC counts to volts" and tell it nothing about our board:

In [ ]:
def counts_to_volts_ai(counts):
    return counts * 3.3 / 1023

check_counts_to_volts(counts_to_volts_ai)

Look at the results.

1. Does the code look wrong when you read it?
2. Which number is the problem? Where could you find the right one?
3. In step 2 we told the AI about our board. This time we didn't. What did the AI do instead?
4. How would you fix it, and how would you check the fix?

*Type your answers below:*



Now change `3.3` to `5.0` in the AI's function and run the cell again. It passes!

But the AI still divides by **1023**, and we divided by **1024**. The test can't tell them apart (the difference is only 0.1%), so which one is right? A test can't answer that. For that, we need to go to the source.

### where does this number come from?

So far we've been using facts about the ADC: it has 10 bits, and a count is the voltage divided by the reference, times 1024. But where do facts like these come from? Not from us, and not from an AI. They come from the **datasheet**: the manufacturer's own document for the chip.

AI can get these details wrong, and when it's about wiring a circuit, a wrong answer can mean a broken circuit. In the labs we had so far, we saw a lot of you asking AI how to wire the circuit up, which is completely fine! But you then will have to double check it against the datasheet, or alternatively you can even ask the AI to tell you exactly where in the datasheet it got the information from. A lot of times the AI got the wiring wrong and when we asked you, you could not point to where the information was taken from.

The datasheet for our chip, the ATmega328P, is long, but you don't need to read it all. You need to know how to find what you're looking for, and you can even use AI to help you do so, just make sure you then double check your answers in the datasheet.

**Try it:** look up the datasheet on the internet (search for the microcontroller's number and "datasheet"). Open the **complete** datasheet, not a short "summary" one: it should be hundreds of pages long. Different versions number their pages and sections differently, so find things by their **names** (Ctrl+F helps), and write down which version (date) you used.

Find the chapter called **Analog-to-Digital Converter**, then answer:

1. What is the formula for the ADC result? Does it use 1023 or 1024? (Hint: search for "ADC Conversion Result".)
2. Which voltage references can the ADC use? (Hint: search for "Voltage Reference Selections".)
3. What is the fastest the ADC can sample, in samples per second? (Hint: look at the Features list at the start of the ADC chapter. If you see two numbers, work out why.)

*My answers:*


Now ask your AI the same three questions, give it the datasheet you downloaded and add this each time: *"Tell me which section and page of the ATmega328P datasheet this comes from."*

For each answer, check:
1. Does it match what **you** found?
2. Is the page or section it gave you real? Open it. Does that page say what the AI says it does?
3. If they don't match, who is right? How do you know?




Now we can convert the whole ECG recording to volts:

In [ ]:
## if you got stuck writing the function before, here is the answer!
def counts_to_volts(counts):
    return counts * 5.0 / 1024

volts = counts_to_volts(counts) # instead of one, call your counts_to_volts function
print(volts[:5])

## Part 4: Look at the ECG

We now have time in seconds and voltage in volts. Plot them, then check the plot against our plan: the flat parts should sit near 2.5 V, and a beat should appear every 0.6 to 1 s.

In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(t, volts)
plt.xlabel("time (s)")
plt.ylabel("voltage (V)")
plt.show()

Look at the plot.

1. Where do the flat parts sit, in volts? Is that what you expected?
2. How many seconds are there between one R wave (the tall spike) and the next? Is that a plausible heart rate?
3. How tall is the R wave, measured up from the flat part? Write it down: we'll need it in Part 5.
4. Is there anything about the line that looks wrong?

## Part 5: Remove the 50 Hz hum

**The goal:** get rid of the 50 Hz ripple, and keep everything that is the heart.

**1. Decide what "right" looks like.** Two things must be true afterwards:
- the 50 Hz ripple is gone
- the R wave is still as tall as the number you wrote down in Part 4

The check we'll use prints both, before and after, so you can compare them with your own number.

**2. Learn.** Ask your AI to explain, not to write code:

> *"My ECG signal is sampled at 250 Hz and has 50 Hz mains hum on it. Don't write any code. What kinds of filter could remove the hum, and what does each kind do to the rest of the signal?"*

Which kind would you choose, and why?

**3. Plan.** One or two lines is enough. Which filter, and what is your check?

*Write your plan:*

**4. Build, then check.** Ask the AI to turn your plan into a function `remove_hum(volts, fs)` that returns the filtered signal. Paste it below (with any `import` lines it needs) and run the check.

In [ ]:
# Paste the AI's function here, replacing the def below. Keep the last line: it runs the check.
def remove_hum(volts, fs):
    return volts

check_hum_removal(remove_hum, volts, fs)

In [ ]:
# Stuck? Run this cell to carry on with a working filter (try it yourself first!)
from scipy.signal import iirnotch, filtfilt

def remove_hum(volts, fs):
    b, a = iirnotch(50, 30, fs=fs)      # notch at 50 Hz
    return filtfilt(b, a, volts)


Now plot the cleaned signal on top of the original:

In [ ]:
clean = remove_hum(volts, fs)

plt.figure(figsize=(9, 3))
plt.plot(t, volts, alpha=0.5, label="before")
plt.plot(t, clean, label="after")
plt.xlabel("time (s)")
plt.ylabel("voltage (V)")
plt.legend()
plt.show()

**5. Now compare with AI's version.** If you ask an AI to "clean up this noisy ECG", it might well give you a low-pass filter, because noise is "fast wiggles". Here's one. Look at the line it gives, then run the same check:

In [ ]:
from scipy.signal import butter, filtfilt

def remove_hum_ai(volts, fs):
    b, a = butter(4, 10, btype="low", fs=fs)     # keep the slow parts, drop everything above 10 Hz
    return filtfilt(b, a, volts)

clean_ai = remove_hum_ai(volts, fs)

plt.figure(figsize=(9, 3))
plt.plot(t, clean_ai)
plt.xlabel("time (s)")
plt.ylabel("voltage (V)")
plt.show()

check_hum_removal(remove_hum_ai, volts, fs)

Look at the results.

1. Is the hum gone? Does the line look clean?
2. What happened to the R wave?
3. Would you have spotted this from the plot alone? What caught it?
4. What would you ask the AI to fix it, and how would you check the fix?

*Type your answers below:*



## Part 6: Find the heart rate

**The goal:** turn the cleaned ECG into one number: the heart rate, in beats per minute.

**1. Decide what "right" looks like.** Two checks:
- it should be plausible for a resting heart (60 to 100 beats per minute)
- it should match what you work out **by hand** from the plot

**Try it before reading on:** in Part 4 you measured the time between two R waves. Turn that into beats per minute (there are 60 seconds in a minute).

**2. Learn.** Ask your AI to explain, not to write code:

> *"I have an ECG recording sampled at 250 Hz, and I want the heart rate in beats per minute. Don't write any code. How would you measure it, and what could go wrong?"*

Did it mention anything you hadn't thought of?

In [ ]:
# The time between two R waves, from your Part 4 answer
gap = None                  # your answer, in seconds
bpm_by_hand = None      # your answer, in beats per minute

**3. Plan.** Write the steps that take you from the cleaned signal to one number, with a check for each.

*Write your plan below:*

**4. Build, then check.** Do it in two small steps. First, find the beats.

SciPy has a function for this: `find_peaks`. Ask your AI what it does and what its settings mean, then write a function `find_beats(signal, fs)` that returns the sample numbers of the R peaks. Paste it below and run the cell. There should be a red dot on every complete R wave, and none anywhere else.

In [ ]:
# Paste the AI's function here, replacing the def below. Keep the last two lines: they show the beats.
from scipy.signal import find_peaks

def find_beats(signal, fs):
    return None

beats = find_beats(clean, fs)
show_beats(t, clean, beats)

In [ ]:
#@title Stuck? Run this cell to carry on with a working find_beats (try it yourself first!)
from scipy.signal import find_peaks

def find_beats(signal, fs):
    peaks, _ = find_peaks(signal, height=np.median(signal) + 0.5)    # the tall peaks are the R waves
    return peaks

beats = find_beats(clean, fs)
show_beats(t, clean, beats)

Now turn the beats into a heart rate. Write a function `heart_rate(signal, fs)` that uses your `find_beats` and returns the heart rate in beats per minute. Run the check as before.

In [ ]:
def heart_rate(signal, fs):
    return None

check_heart_rate(heart_rate, clean, fs, bpm_by_hand)

**5. Now compare with AI's version.** The simplest way to get a heart rate is to count the beats and divide by the length of the recording. An AI might well give you this:

In [ ]:
from scipy.signal import find_peaks

def heart_rate_ai(signal, fs):
    peaks, _ = find_peaks(signal, height=3.0)     # beats are the tall peaks
    duration = len(signal) / fs                   # length of the recording, in seconds
    return len(peaks) / duration * 60             # beats per second, times 60

check_heart_rate(heart_rate_ai, clean, fs, bpm_by_hand)

Look at the results.

1. Is the answer plausible? Does it match your by-hand answer?
2. Would a "plausible" check alone have caught it?
3. How many beats did it count? Look at your plot of the beats: how many R waves are there, and what happens at the right-hand edge? Why does that make the answer wrong?
4. What would you ask the AI to fix it, and how would you check the fix?

*Type your answers:*



## Wrap-up

For every part, you did the same four things:

1. **Decided what "right" looks like** before writing any code
2. **Learned** the idea (asked the AI to explain or quiz you, not to write)
3. **Planned** it, with a check for each step
4. **Built it, then checked it**, and compared it with a plausible-looking AI version

Every AI version ran, and most looked believable. Here is what was wrong, and what caught it:

| Part | What was wrong | What caught it |
|---|---|---|
| Prime numbers | called 4, 6, 8, 9, 15 and 25 prime | the primes you listed before coding |
| Time axis | last sample at 2.000 s, not 1.996 s | the numbers you worked out by hand |
| Counts to volts | assumed 3.3 V, not 5 V | the 2.000 V reference |
| Remove the hum | the R wave lost more than half its height | the R-wave height before and after |
| Heart rate | 60 bpm, not 75 | the heart rate you worked out by hand |

Each time, the thing that caught the problem was something **you decided before asking the AI**.

Two more habits:
- When a step is too big, **make it smaller**.
- For facts about a chip, **go to the datasheet**. The AI is a helper, not the source.

**Explain it back.** Pick one part. Explain every line of your code to the person next to you, without looking at the AI's explanation. If you can't, you're not ready to trust it.